# Lab 2 · Advanced RAG over the real Kubernetes docs (free Kaggle T4)

**Layer 3 · Data — Chapter 2 (the advanced pipeline).** Lab 1 used a tidy Q/A set.
Here we point the pipeline at the **real k8s documentation** — hundreds of messy
markdown pages — and add the four levers that make retrieval *good*, then **measure**
them with an **independent** eval set:

1. **128-token chunking** of real docs (Phase 2 cleaning for real)
2. **Metadata filtering** — restrict by doc area
3. **Hybrid search** — fuse vector (meaning) with BM25 (exact terms)
4. **Reranking** — a cross-encoder re-scores query+chunk together

The chunk/index/eval core is **CPU-only and runs on any GPU**. The grounded LLM answer
(last cell) needs a **T4**.

**Settings:** Accelerator = **GPU T4 x2** (not P100), Internet = **On**, **Run All**.

*Corpus: [`kubernetes/website`](https://github.com/kubernetes/website) `content/en/docs/{tasks,concepts}` (pinned commit).*

In [ ]:
import warnings, logging, os
warnings.filterwarnings('ignore')
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['HF_HUB_VERBOSITY'] = 'error'
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
for _n in ('huggingface_hub', 'sentence_transformers', 'transformers'):
    logging.getLogger(_n).setLevel(logging.ERROR)

!pip install -q faiss-cpu sentence-transformers rank-bm25 2>/dev/null

import re, glob, time, textwrap
import numpy as np, faiss
from collections import defaultdict, Counter
from sentence_transformers import SentenceTransformer, CrossEncoder

## 0 · Ingest real docs

Clone the Kubernetes website repo (pinned commit, sparse checkout of `tasks` + `concepts`). Hundreds of real markdown pages — no tidy Q/A pairs.

In [ ]:
# Real docs, not a toy Q/A set: clone the Kubernetes website repo (pinned commit for
# reproducibility), sparse-checkout just the tasks + concepts sections.
SHA = '54c74114c3a688d3577168c27fff7a7597b96c06'
!rm -rf /tmp/k8s-site
!git clone -q --depth 1 --filter=blob:none --sparse https://github.com/kubernetes/website.git /tmp/k8s-site
!cd /tmp/k8s-site && git sparse-checkout set content/en/docs/tasks content/en/docs/concepts 2>/dev/null

ROOT = '/tmp/k8s-site/content/en/docs'
files = glob.glob(f'{ROOT}/tasks/**/*.md', recursive=True) + glob.glob(f'{ROOT}/concepts/**/*.md', recursive=True)
print('markdown files:', len(files))

## 1 · Clean + chunk (Phase 2, for real)

Real docs are messy: YAML front-matter, Hugo shortcodes `{{< … >}}`, HTML comments. We strip those (keeping the `title`), then split each page into **128-token chunks** using the embedder's own tokenizer. Each chunk keeps its `source` and `title` as metadata.

In [ ]:
# Phase 2 for real: strip YAML front-matter (keep the title), Hugo shortcodes
# ({{< glossary_tooltip >}}), HTML comments/tags — then split into 128-TOKEN chunks.
FM = re.compile(r'^---\n(.*?)\n---\n', re.S)
SC = re.compile(r'\{\{[<%].*?[%>]\}\}', re.S)   # Hugo shortcodes
CM = re.compile(r'<!--.*?-->', re.S)               # HTML comments
HT = re.compile(r'<[^>]+>')                         # stray HTML tags

def clean(t):
    t = SC.sub('', t); t = CM.sub('', t); t = HT.sub('', t)
    return re.sub(r'\n{3,}', '\n\n', t).strip()

embedder = SentenceTransformer('BAAI/bge-small-en-v1.5', device='cpu')
tokz = embedder.tokenizer
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '

def token_chunks(text, size=128, overlap=24):
    # slice the ORIGINAL text at real token boundaries (offset_mapping) — clean
    # substrings, no decode artifacts, still ~128-token windows.
    offs = tokz(text, add_special_tokens=False, return_offsets_mapping=True)['offset_mapping']
    step = size - overlap
    out = []
    for i in range(0, len(offs), step):
        w = offs[i:i+size]
        if w: out.append(text[w[0][0]:w[-1][1]].strip())
    return [c for c in out if c]

chunks = []
for f in files:
    raw = open(f, encoding='utf-8').read()
    m = FM.match(raw)
    title = None
    if m:
        mt = re.search(r'^title:\s*(.+)$', m.group(1), re.M)
        title = mt.group(1).strip().strip('"\'') if mt else None
    body = clean(raw[m.end():] if m else raw)
    if len(body.split()) < 20:
        continue
    src = f.replace(ROOT + '/', '')
    for j, ch in enumerate(token_chunks(body)):
        chunks.append({'id': f'{src}#{j}', 'text': ch, 'title': title, 'source': src})

print('chunks:', len(chunks), '| avg tokens/chunk ~128')
c = chunks[20]
print('\nsample  source:', c['source'], '| title:', c['title'])
print(' text:', c['text'][:240])

## 2 · Embed + index

Every chunk → a vector via **`bge-small`** (CPU), stored in a flat **FAISS** index (exact cosine; HNSW only past ~100k). The query uses the **same** embedder.

In [ ]:
t0 = time.time()
vecs = embedder.encode([c['text'] for c in chunks], normalize_embeddings=True,
                       convert_to_numpy=True, batch_size=128).astype('float32')
index = faiss.IndexFlatIP(vecs.shape[1]); index.add(vecs)

def vector(query, k=5):
    qv = embedder.encode([QUERY_PREFIX + query], normalize_embeddings=True,
                         convert_to_numpy=True).astype('float32')
    sc, ii = index.search(qv, k)
    return [{**chunks[int(i)], 'score': float(s)} for s, i in zip(sc[0], ii[0])]

print(f'embedded + indexed {index.ntotal} chunks on CPU in {time.time()-t0:.0f}s')
for h in vector('how do I give a container a memory limit?', 3):
    print(f"  {h['score']:.3f}  {h['source']}")

## 3 · Metadata filtering

We derive an `area` from each source path (e.g. `tasks/administer-cluster`). Filtering at retrieval is what enforces *permissions* and *freshness* in production. Flat FAISS has no native filter, so we over-fetch then filter (a real vector DB does this inside the search).

In [ ]:
# Each chunk carries metadata (source path -> we derive an 'area'). Filtering at
# retrieval is what enforces permissions / freshness in production.
for c in chunks: c['area'] = '/'.join(c['source'].split('/')[:2])
print('top areas:', Counter(c['area'] for c in chunks).most_common(5))

def vector_filtered(query, k=5, area=None):
    qv = embedder.encode([QUERY_PREFIX + query], normalize_embeddings=True,
                         convert_to_numpy=True).astype('float32')
    sc, ii = index.search(qv, 80)
    hits = [{**chunks[int(i)], 'score': float(s)} for s, i in zip(sc[0], ii[0])]
    if area: hits = [h for h in hits if h['area'] == area]
    return hits[:k]

q = 'how do I limit resource usage?'
print(f"\nno filter          :", [h['area'] for h in vector(q, 5)])
print(f"area='tasks/administer-cluster':", [h['source'].split('/')[-1] for h in vector_filtered(q, 5, area='tasks/administer-cluster')])

## 4 · Hybrid search (vector ∪ BM25)

Dense vectors capture *meaning* but can under-rank **exact terms** (`sysctl`, `kubectl`, error codes). **BM25** is keyword search. We fuse the two rankings with **Reciprocal Rank Fusion**.

In [ ]:
from rank_bm25 import BM25Okapi
bm25 = BM25Okapi([c['text'].lower().split() for c in chunks])

def hybrid(query, k=5, depth=30):
    qv = embedder.encode([QUERY_PREFIX + query], normalize_embeddings=True,
                         convert_to_numpy=True).astype('float32')
    _, di = index.search(qv, depth); dense = [int(i) for i in di[0]]
    bm = list(np.argsort(bm25.get_scores(query.lower().split()))[::-1][:depth])
    rrf = defaultdict(float)                       # Reciprocal Rank Fusion
    for r, i in enumerate(dense): rrf[i] += 1 / (60 + r)
    for r, i in enumerate(bm):    rrf[i] += 1 / (60 + r)
    return [{**chunks[i], 'rrf': rrf[i]} for i in sorted(rrf, key=rrf.get, reverse=True)[:k]]

# A query with an exact term ('sysctl') that BM25 nails:
q = 'how do I set sysctl kernel parameters for pods?'
print('vector-only top-3:', [h['source'].split('/')[-1] for h in vector(q, 3)])
print('hybrid    top-3:', [h['source'].split('/')[-1] for h in hybrid(q, 3)])

## 5 · Reranking

Vector/BM25 are fast but blunt. A **cross-encoder** reads query+chunk *together* and re-scores. Over-fetch a deep pool, then keep the best few — the biggest lever after chunking.

In [ ]:
reranker = CrossEncoder('BAAI/bge-reranker-base', device='cpu')

q = 'how do I roll out a new version of my app without downtime?'
cands = vector(q, 30)                                       # over-fetch a deep pool
print('BEFORE rerank (vector order), top 3:')
for h in cands[:3]: print('  ', h['source'])
s = reranker.predict([[q, h['text']] for h in cands])
for h, v in zip(cands, s): h['rr'] = float(v)
print('AFTER rerank (cross-encoder), top 3:')
for h in sorted(cands, key=lambda h: h['rr'], reverse=True)[:3]:
    print(f"   rr={h['rr']:.2f}  {h['source']}")

## 6 · Does it actually help? — evaluate on an independent set

The honest test: **40 hand-written user questions**, each mapped to the doc that owns the answer (`gold_source`) — *not* derived from the chunks, so it can't flatter retrieval (the circular 'answer is its own gold' eval lies). We score **hit@3** and **MRR** for vector-only → + hybrid → vector + rerank.

Read it honestly: **hybrid gives a small lift** (real exact terms like `sysctl` help); **reranking is marginal here** — a cross-encoder wants more context than a 128-token chunk carries. The pipeline is **deterministic** (greedy, fixed corpus), so reruns give the *same* table — but it's only **40 questions** (≈2.5pts each) over *one* corpus: read the **direction**, not the third decimal, and don't generalize past this data. The numbers move when you change the *chunking* or the *eval set*, not when you rerun — which is exactly why you measure on **your** corpus.

In [ ]:
# Independent eval set: 40 natural user questions, each hand-mapped to the doc that
# owns the answer (gold_source). NOT derived from chunk text — so it doesn't flatter
# retrieval the way 'the answer is its own gold chunk' would. hit@3 = a chunk from the
# right doc in the top 3; MRR = 1 / rank of the first such chunk.
EVAL = [{'q': 'How do I give a container a memory request and limit?', 'gold_source': 'tasks/configure-pod-container/assign-memory-resource.md'}, {'q': "What's the way to cap how much CPU a pod can use?", 'gold_source': 'tasks/configure-pod-container/assign-cpu-resource.md'}, {'q': 'How can I pull an image from a private registry?', 'gold_source': 'tasks/configure-pod-container/pull-image-private-registry.md'}, {'q': 'How do I add health checks so Kubernetes restarts a stuck container?', 'gold_source': 'tasks/configure-pod-container/configure-liveness-readiness-startup-probes.md'}, {'q': 'How do I mount a ConfigMap into my pod?', 'gold_source': 'tasks/configure-pod-container/configure-pod-configmap.md'}, {'q': "How do I open a shell inside a container that's already running?", 'gold_source': 'tasks/debug/debug-application/get-shell-running-container.md'}, {'q': 'My pod keeps crashing, how do I troubleshoot it?', 'gold_source': 'tasks/debug/debug-application/debug-pods.md'}, {'q': 'How can I cordon and drain a node before maintenance?', 'gold_source': 'tasks/administer-cluster/safely-drain-node.md'}, {'q': 'How do I upgrade a cluster built with kubeadm?', 'gold_source': 'tasks/administer-cluster/kubeadm/kubeadm-upgrade.md'}, {'q': 'How do I encrypt secret data at rest in etcd?', 'gold_source': 'tasks/administer-cluster/encrypt-data.md'}, {'q': 'How do I expose my app to the internet with a load balancer?', 'gold_source': 'tasks/access-application-cluster/create-external-load-balancer.md'}, {'q': 'How do I forward a local port to a pod for testing?', 'gold_source': 'tasks/access-application-cluster/port-forward-access-application-cluster.md'}, {'q': 'How do I autoscale a deployment based on CPU usage?', 'gold_source': 'tasks/run-application/horizontal-pod-autoscale-walkthrough.md'}, {'q': 'How can I roll out a new version without downtime?', 'gold_source': 'tasks/run-application/update-deployment-rolling.md'}, {'q': 'How do I run a job on a recurring schedule?', 'gold_source': 'tasks/job/automated-tasks-with-cron-jobs.md'}, {'q': 'How do I install kubectl on Linux?', 'gold_source': 'tasks/tools/install-kubectl-linux.md'}, {'q': 'How do I create a Secret using kubectl?', 'gold_source': 'tasks/configmap-secret/managing-secret-using-kubectl.md'}, {'q': 'How do I block traffic between pods with a NetworkPolicy?', 'gold_source': 'tasks/administer-cluster/declare-network-policy.md'}, {'q': 'How can I schedule GPUs to my workloads?', 'gold_source': 'tasks/manage-gpus/scheduling-gpus.md'}, {'q': 'How do I set a securityContext for a pod or container?', 'gold_source': 'tasks/configure-pod-container/security-context.md'}, {'q': 'How do I pin pods to particular nodes using node affinity?', 'gold_source': 'tasks/configure-pod-container/assign-pods-nodes-using-node-affinity.md'}, {'q': 'How do I turn on automatic certificate rotation for the kubelet?', 'gold_source': 'tasks/tls/certificate-rotation.md'}, {'q': 'How do I set environment variables for a container?', 'gold_source': 'tasks/inject-data-application/define-environment-variable-container.md'}, {'q': 'How do I manage several clusters and switch contexts?', 'gold_source': 'tasks/access-application-cluster/configure-access-multiple-clusters.md'}, {'q': 'DNS lookups are failing inside my pods, how do I debug that?', 'gold_source': 'tasks/administer-cluster/dns-debugging-resolution.md'}, {'q': 'How do I cap total CPU and memory for a whole namespace?', 'gold_source': 'tasks/administer-cluster/manage-resources/quota-memory-cpu-namespace.md'}, {'q': 'How do I make sure enough replicas stay up during disruptions?', 'gold_source': 'tasks/run-application/configure-pdb.md'}, {'q': 'How can I tell which container runtime a node is using?', 'gold_source': 'tasks/administer-cluster/migrating-from-dockershim/find-out-runtime-you-use.md'}, {'q': 'How do I run a DaemonSet on only a subset of nodes?', 'gold_source': 'tasks/manage-daemon/pods-some-nodes.md'}, {'q': 'How do two containers in the same pod share files?', 'gold_source': 'tasks/access-application-cluster/communicate-containers-same-pod-shared-volume.md'}, {'q': 'How do I create a static pod the kubelet manages directly?', 'gold_source': 'tasks/configure-pod-container/static-pod.md'}, {'q': 'How do I override the entrypoint command and arguments of a container?', 'gold_source': 'tasks/inject-data-application/define-command-argument-container.md'}, {'q': 'How do I scale the number of replicas in a StatefulSet?', 'gold_source': 'tasks/run-application/scale-stateful-set.md'}, {'q': 'How do I enforce Pod Security Standards using namespace labels?', 'gold_source': 'tasks/configure-pod-container/enforce-standards-namespace-labels.md'}, {'q': 'How do I list every container image running across the cluster?', 'gold_source': 'tasks/access-application-cluster/list-all-running-container-images.md'}, {'q': 'How do I set kernel sysctl parameters for pods?', 'gold_source': 'tasks/administer-cluster/sysctl-cluster.md'}, {'q': 'How do I extend the Kubernetes API with a custom resource?', 'gold_source': 'tasks/extend-kubernetes/custom-resources/custom-resource-definitions.md'}, {'q': 'How do I attach a service account to my pods?', 'gold_source': 'tasks/configure-pod-container/configure-service-account.md'}, {'q': 'How do I reserve compute resources for system daemons on a node?', 'gold_source': 'tasks/administer-cluster/reserve-compute-resources.md'}, {'q': 'How do I patch a live object in place with kubectl?', 'gold_source': 'tasks/manage-kubernetes-objects/update-api-object-kubectl-patch.md'}]

def rank_of(gold, ordered):
    for r, h in enumerate(ordered, 1):
        if h['source'] == gold: return r
    return None

def score(retriever, depth_rerank=False):
    rr = hit3 = 0
    for it in EVAL:
        hits = retriever(it['q'])
        r = rank_of(it['gold_source'], hits)
        if r: rr += 1 / r; hit3 += (r <= 3)
    n = len(EVAL); return hit3 / n, rr / n

def vec(q):  return vector(q, 30)
def hyb(q):  return hybrid(q, 30)
def rer(q):
    cands = vector(q, 30)
    s = reranker.predict([[q, h['text']] for h in cands])
    for h, v in zip(cands, s): h['rr'] = float(v)
    return sorted(cands, key=lambda h: h['rr'], reverse=True)

print(f'over {len(EVAL)} questions       hit@3    MRR')
for name, fn in [('vector-only', vec), ('+ hybrid (BM25)', hyb), ('vector + rerank', rer)]:
    h, m = score(fn); print(f'  {name:<18} {h:5.2f}   {m:5.2f}')

## 7 · Grounded, cited answer (needs T4)

Retrieve → rerank → keep top chunks → **Qwen2.5-3B** answers *only* from them. The eval above scores *retrieval*; this is the user-facing half — and a user reads a **doc title + a kubernetes.io link**, not a raw `tasks/…/foo.md` path, so we cite that. Loads the LLM (~6 GB); needs a **T4**. On P100 it's skipped with a note and the eval above still stands.

In [ ]:
# Grounded, cited answer over the REAL docs. Needs the GPU LLM -> a **T4** runtime
# (Kaggle's P100 can't run the prebuilt fp16 kernels). Wrapped so the notebook still
# completes on P100 — the eval above is the deliverable; this is the demo on top.
try:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map='auto').eval()
    _ = model.generate(**tok('ok', return_tensors='pt').to(model.device), max_new_tokens=2)  # P100 fails here

    # A real user reads a doc TITLE + link, not 'tasks/.../assign-memory-resource.md'.
    def doc_url(src):                                   # repo path -> live kubernetes.io URL
        p = src[:-9] if src.endswith('_index.md') else src[:-3]   # drop _index.md / .md
        return 'https://kubernetes.io/docs/' + p.rstrip('/') + '/'
    def cite(h):                                        # human label the model puts in [brackets]
        return h['title'] or h['source'].split('/')[-1]

    SYS = ('You are a Kubernetes assistant. Use ONLY the CONTEXT to answer, and cite the source '
           'titles in [brackets]. '
           "If the answer is not in the CONTEXT, reply exactly: I don't know.")
    def answer(query, keep=4):
        cands = vector(query, 30)
        s = reranker.predict([[query, h['text']] for h in cands])
        for h, v in zip(cands, s): h['rr'] = float(v)
        hits = sorted(cands, key=lambda h: h['rr'], reverse=True)[:keep]
        ctx = '\n'.join(f"[{cite(h)}] {h['text']}" for h in hits)
        text = tok.apply_chat_template([{'role': 'system', 'content': SYS},
                {'role': 'user', 'content': f'CONTEXT:\n{ctx}\n\nQUESTION: {query}'}],
                tokenize=False, add_generation_prompt=True)
        inp = tok(text, return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=220, do_sample=False, pad_token_id=tok.eos_token_id)
        return tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip(), hits

    a, hits = answer('How do I give a container a memory request and limit?')
    srcs = list(dict.fromkeys((cite(h), doc_url(h['source'])) for h in hits))   # dedup, keep order
    print('SOURCES (what a user sees, not raw .md paths):')
    for title, url in srcs: print(f'  • {title} — {url}')
    print('\nANSWER:\n', textwrap.fill(a, 90))
    print('\nREFUSAL:', answer('What is the capital of France?')[0])
except Exception as e:
    print('LLM step skipped:', type(e).__name__, '-', str(e)[:120])
    print('>> Needs a T4 runtime: Settings -> Accelerator = GPU T4 x2 (not P100), then Run All.')

## Takeaways

- **Real docs need real Phase 2.** Front-matter, shortcodes, HTML — clean before you
  chunk, or you embed noise. 128-token chunks trade context for precision.
- **Measure on an *independent* set.** Questions mapped to gold docs (not to the chunks
  themselves) give honest hit@3 / MRR — the circular "answer is its own gold" eval lies.
- **Each lever is conditional.** Hybrid gave a small consistent win here; reranking was
  marginal (128-token chunks are short for a cross-encoder); metadata filtering is a
  correctness/permissions must regardless of the scores. The table tells you which pays
  off *on this corpus* — the generalization "rerank is the biggest lever" (Chapter 2) is
  true *often*, not *always*.
- **Small, corpus-specific eval.** 40 questions over one corpus → coarse (≈2.5pts each)
  and not generalizable. It's deterministic (reruns match), but it shifts with chunking
  or the eval set — trust the direction, grow the set before you over-claim.
- Next: agentic RAG — the agent *decides* whether/what/when to retrieve (Layer 4).